# M2 · Evaluation harness — MacGyver Gym Rat

**SI4006 · Universidad EAFIT · Module 2** · runs end to end on a free Colab T4.

This notebook measures the M1 system on three dimensions and writes
`reports/scorecard_baseline.csv`. It does not train anything.

| Dimension | What it asks | Where it lives |
|---|---|---|
| 1 · classic metric | Does the answer *look* like a valid one? Embedding cosine + ROUGE-L of the recited steps. | `eval/harness.py` |
| 2 · LLM-as-a-judge | Is it correct, safe and appropriate? 1-5 against a versioned rubric. | `eval/rubric.md` |
| 3 · domain hit rate | Does the catalog agree the exercise is real, for that muscle, with that object — and does the system refuse when it should? | `eval/harness.py` |

Two systems get the same yardstick: the base model with no adapter (zero-shot)
and the M1 LoRA. In M3 the retrieval system becomes a third column, and nothing
in this notebook changes except the callable passed to `harness()`.

**Runtime → Change runtime type → T4 GPU.** On CPU this finishes, but slowly.


## 0 · Setup

Clone the repo *with the dataset submodule* — the catalog is what grades dimension 3, and without it nothing below runs.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

REPO = "https://github.com/iamcroody/models-for-exercises-dataset.git"
# The M2 harness is on main; the branch stays a variable so a fork or a working
# branch can be run through this same notebook without editing anything else.
BRANCH = "main"
ROOT = Path("/content/models-for-exercises-dataset")

if "google.colab" in sys.modules:
    if not ROOT.exists():
        clone = ["git", "clone", "--recurse-submodules"]
        try:
            subprocess.run(clone + ["-b", BRANCH, REPO, str(ROOT)], check=True)
        except subprocess.CalledProcessError:
            # The branch was renamed or removed. Fall back to the default branch
            # rather than dying on a pointer that went stale after the fact.
            print(f"branch {BRANCH!r} not found, cloning the default branch")
            shutil.rmtree(ROOT, ignore_errors=True)
            subprocess.run(clone + [REPO, str(ROOT)], check=True)
    os.chdir(ROOT)
else:
    # Running from the repo itself (local, or `jupyter` in the project). Walk up
    # from the current directory until the repo root shows itself, so this works
    # from notebooks/ or from anywhere below it.
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "eval/harness.py").exists()), Path.cwd())
    os.chdir(ROOT)

print("working in", Path.cwd())
# RuntimeError, not SystemExit: IPython treats SystemExit as the end of the cell
# and "Run all" keeps going, so the real problem would resurface three cells
# later as an unreadable ModuleNotFoundError.
if not (Path.cwd() / "eval/harness.py").exists():
    raise RuntimeError("eval/ is missing: clone the repo at a commit that carries it")
if not (Path.cwd() / "data/exercises-dataset/data/exercises.json").exists():
    raise RuntimeError("the dataset submodule is missing: "
                       "git submodule update --init --recursive")

In [ ]:
# Colab ships torch and transformers, but both `mg.load_model` and the judge pass
# `dtype=` to `from_pretrained`, which the preinstalled transformers is too old to
# accept. So transformers is upgraded rather than trusted — the same recipe M1 uses,
# including the torchao removal, so both modules run on the same stack.
# The versions are pinned: a rerun months from now should grade with the same code
# the numbers in the README came from.
%pip install -q transformers==5.16.1 peft==0.20.0
# sentence-transformers is deliberately unpinned — no version has been verified
# against this notebook yet. Pin the number printed below once a full run comes
# back clean, and this list is closed.
%pip install -q sentence-transformers rouge-score==0.1.2
# Colab pins torchao 0.10 to its torch build; peft refuses anything below 0.16,
# and upgrading it drags a different torch in behind it.
%pip uninstall -y -q torchao
# Optional: put HF_TOKEN in the Colab secrets panel. Nothing used here is gated,
# but an authenticated session pulls the ~7 GB of weights noticeably faster.

import torch, transformers, peft, sentence_transformers
print(f"torch {torch.__version__} | transformers {transformers.__version__} | "
      f"peft {peft.__version__} | sentence-transformers {sentence_transformers.__version__}")

# An unattended restart has to fail here, with instructions, rather than as a
# TypeError on `dtype=` six cells down.
assert transformers.__version__.startswith("5.16"), (
    "the pre-installed transformers is still loaded. Runtime > Restart session, "
    "then Runtime > Run all. Cell 2 is idempotent (`if not ROOT.exists()`), so "
    "the rerun costs seconds, not another clone.")

In [ ]:
import json, gc, torch
sys.path.insert(0, "scripts")
sys.path.insert(0, "eval")

import macgyver as mg
import harness as H

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)
print("rubric:", H.load_rubric()["version"], "| seed:", H.SEED)

## 1 · The eval set

Fourteen examples, four of them adversarial (29%). Built by `eval/build_eval_set.py`
from the catalog, so every reference answer's steps are the real ones; the
(muscle, object) pairs were chosen by hand and none of them appears in the M1
training split.

Run `python eval/build_eval_set.py` to rebuild it — the script refuses to write a
contaminated or unanswerable example.

In [ ]:
eval_set = H.load_eval_set()

print(f"{len(eval_set)} examples, "
      f"{sum(e['adversarial'] for e in eval_set)} adversarial\n")
for e in eval_set:
    meta = e["meta"]
    tag = "ADV" if e["adversarial"] else "   "
    obj = (meta.get("object") or "-")[:34]
    print(f"{tag} {e['id']:26} {e['kind']:14} {str(meta.get('target')):12} {obj}")

In [ ]:
# Sanity check with no model weights: a system that replies with the reference
# answer must score 1.0 on dimension 3, and one that always says "plank" must
# score 0.0. If this fails, every number below is meaningless.
H.self_test();

## 2 · Generating the replies

All three models fit on the T4 at once — the generator, the judge and the
embedding model come to about 7 GB of its 15 GB — so this order is a choice, not
a constraint. Generating first, freeing the GPU, then judging keeps the peak near
4 GB, and that headroom is what M3's retriever moves into without this notebook
changing. The harness sees a callable either way (`H.replay`), so the measurement
is identical under both arrangements. The resource that is actually tight on the
free tier is system RAM (12.7 GB), not VRAM.

Greedy decoding, `enable_thinking=False`, the same 448-token budget M1 used.

In [ ]:
def generate(adapter=None, label=""):
    tok = mg.load_tokenizer()
    model = mg.load_model(mg.BASE_MODEL, adapter=adapter)
    system = H.make_hf_system(model, tok)

    replies = {}
    for i, e in enumerate(eval_set, 1):
        replies[e["input"]] = system(e["input"])
        print(f"  {label} {i}/{len(eval_set)}", end="\r", flush=True)
    # The carriage returns above collapse when the notebook is saved, so the
    # progress line leaves no evidence that generation ran. This one does.
    print(f"  {label}: {len(replies)} replies")

    # `system` closes over model and tok, so deleting only those two names frees
    # nothing and empty_cache runs with the weights still referenced — which is
    # how the second generate() call OOMs on a free T4.
    del system, model, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return replies

replies_zeroshot = generate(adapter=None, label="zero-shot")
replies_lora = generate(adapter="models/r16-all-linear", label="lora")

Path("reports").mkdir(exist_ok=True)
Path("reports/replies_m2.json").write_text(json.dumps(
    {"zeroshot": replies_zeroshot, "lora": replies_lora}, indent=2, ensure_ascii=False))
print("replies cached in reports/replies_m2.json")

In [ ]:
# One example, to see what we are actually grading.
sample = eval_set[4]
print(sample["input"][:180], "...\n")
print("--- LoRA reply ---")
print(replies_lora[sample["input"]][:700])

## 3 · Dimension 1 — the classic metric

Cosine similarity between sentence embeddings, plus ROUGE-L of the recited steps
against the real steps of the exercise the model *named*.

Why both: similarity alone rewards an answer that reads like the reference, and
in this domain "reads like it" and "is safe to do" are different questions. ROUGE
is scored against the named exercise rather than the reference exercise, so
picking a different but valid exercise is not punished.

In [ ]:
similarity = H.Similarity()

print("paraphrase :", round(similarity("The cat sleeps.", "The feline is resting."), 2))
print("unrelated  :", round(similarity("The cat sleeps.", "The car is red."), 2))

## 4 · Dimension 2 — the judge

`Qwen/Qwen2.5-1.5B-Instruct`, small enough for the free tier, scoring 1-5 against
`eval/rubric.md` (rubric **v1.0**). The rubric text is read from that file at
runtime, so a score can always be traced to the exact wording that produced it.

The parser returns `None` rather than a neutral 3 when the judge fails to emit a
digit: a judge that quietly scores 3 whenever it rambles looks like an average
judge instead of a broken one.

In [ ]:
judge = H.Judge(device=DEVICE)
print("judge:", judge.model_id, "| rubric:", judge.rubric["version"])
print()
print(judge.rubric["pointwise"][:600], "...")

The excerpt above is the first 600 characters of the pointwise block, roughly
two fifths of it, and it never reaches the pairwise block at all — the part of
the rubric that governs the position probe in section 5 and
`judge.compare_robust`. The whole file follows, read from the repo at run time:
this is the versioned text every score in this notebook traces back to.

In [ ]:
print(Path("eval/rubric.md").read_text())

In [ ]:
# Does the judge separate a good answer from a bad one at all? If it does not,
# dimension 2 is noise and should be reported as such.
case = eval_set[4]
good = case["esperado"]
poor = ("Exercise: Mega Brick Curl 3000\nGym equivalent: barbell\n"
        "Adaptation: Tie the bricks to your wrists with a belt and swing hard.\n"
        "Steps:\n1. Swing as fast as you can for two minutes.\n"
        "Safety: No need to warm up.")

print("good answer ->", judge.score(case["input"], good, case["esperado"]), "/ 5")
print("poor answer ->", judge.score(case["input"], poor, case["esperado"]), "/ 5")

## 5 · Judge bias — measured, then mitigated

Two known biases, both probed on our own data rather than cited from a paper.

**Position bias.** Ask the judge to pick the better of two answers, then ask
again with the answers swapped. Every disagreement is a verdict that came from
the seating order, not the content. Mitigation: `judge.compare_robust` asks both
ways and only declares a winner when the two agree — otherwise it is a tie.

**Length / verbosity bias.** Score each reply, then score it again padded with
content-free filler. Any positive mean delta is the judge paying for words. The
rubric says in as many words that length is not quality, and the judge sees at
most `MAX_ANSWER_CHARS` of an answer — but the cap is a candidate mitigation, so
the probe below reports the delta with the cap lifted and with it applied instead
of asserting that it works. On this data it does not mitigate anything. The
filler is prepended, so the cap trims the tail of the answer and never the
padding, and only one of the fourteen padded answers is long enough to be cut at
all: `adv03`, at 2395 characters, whose unpadded score was 1 rather than 5.

In [ ]:
gold_replies = [replies_lora[e["input"]] for e in eval_set]

print("position bias, reference answer vs LoRA reply, both orders:")
position = H.position_bias_probe(judge, eval_set, gold_replies)
print(f"\nflip rate: {position['flip_rate']:.0%} "
      f"({position['flips']} of {position['decided']} decided verdicts "
      f"depended on the order)")
print(f"unparsed: {position['unparsed']} of {position['n']} pairs, kept out of "
      f"the rate so 'gave no letter' is not counted as 'changed its mind'")


In [ ]:
print("length bias, same reply padded with filler:")
length = H.length_bias_probe(judge, eval_set, gold_replies)
print(f"\nmean delta, cap lifted : {length['mean_delta_uncapped']} "
      f"({length['raised_uncapped']} raised, {length['lowered_uncapped']} lowered)")
print(f"mean delta, cap applied: {length['mean_delta_capped']} "
      f"({length['raised_capped']} raised, {length['lowered_capped']} lowered)")
print(f"\nthe first number is the bias, the second is what survives the "
      f"{length['max_answer_chars']}-character cap.")


In [ ]:
# The mitigation in use: a robust pairwise verdict on the same pairs.
verdicts = {"x": 0, "y": 0, "tie": 0}
for e, reply in zip(eval_set, gold_replies):
    verdicts[judge.compare_robust(e["input"], e["esperado"], reply)] += 1
print("reference wins:", verdicts["x"], "| LoRA wins:", verdicts["y"],
      "| undecided after both orders:", verdicts["tie"])

## 6 · The scorecard

`harness(eval_set, system)` runs all three dimensions over one callable. Two
callables here; in M3 the RAG pipeline is a third.

In [ ]:
print("zero-shot")
sc_zeroshot = H.harness(eval_set, H.replay(replies_zeroshot), judge=judge,
                        similarity=similarity, label="zeroshot")
print("\nLoRA (M1)")
sc_lora = H.harness(eval_set, H.replay(replies_lora), judge=judge,
                    similarity=similarity, label="lora_m1")

H.print_scorecard([sc_zeroshot, sc_lora])
print("judge outputs that did not parse:", judge.unparsed)

In [ ]:
path = H.write_scorecard([sc_zeroshot, sc_lora], "reports/scorecard_baseline.csv")
Path("reports/scorecard_baseline_detail.json").write_text(json.dumps(
    {"zeroshot": sc_zeroshot, "lora_m1": sc_lora,
     "position_bias": position, "length_bias": length},
    indent=2, ensure_ascii=False))
print("wrote", path, "and reports/scorecard_baseline_detail.json")
print(path.read_text())

## 7 · Where it fails

The list below is what goes into the README's honest reading. Read it before
writing that paragraph, not after.

In [ ]:
for name, sc in (("zero-shot", sc_zeroshot), ("LoRA", sc_lora)):
    print(f"\n=== {name}: {sc['domain_hits']}/{sc['n']} domain hits ===")
    for row in sc["detail"]:
        if not row["hit"]:
            print(f"\n  {row['id']} ({row['kind']}) — {row['why_missed']}")
            print(f"    judge={row['judge']} sim={row['similarity']}")
            print("    " + " ".join(row["reply"].split())[:220])

## 8 · Grading the judge

Dimension 3 never reads the answer. It takes the exercise the system named and
resolves it against the catalog, so it has no opinion that could be wrong — which
makes it a label rather than a second opinion, and a label makes the judge
measurable. Every row already carries both numbers: a 1-5 judge score and a
binary verdict on whether the answer was any good. Asking whether the first
predicts the second turns dimension 2 from something we assert about into
something we report.

Pooled over both systems, because per system the question is degenerate: a column
with no domain hits has no positive class, and an AUC needs one of each.

In [ ]:
q = H.pooled_judge_quality([sc_zeroshot, sc_lora])

print(f"pooled rows       : {q['n']}  ({q['n_hits']} domain hits, {q['n_misses']} misses)")
print(f"mean on hits      : {q['mean_on_hits']}")
print(f"mean on misses    : {q['mean_on_misses']}")
print(f"gap               : {q['gap']}   (how far apart the judge puts the two classes)")
print(f"AUC               : {q['auc']}   (0.500 = no information, 1.000 = every hit "
      f"scored above every miss)")
print(f"point-biserial r  : {q['point_biserial_r']}")

c = q["confusion"]
print(f"\nreading \"judge >= {q['threshold']}\" as the judge's own verdict of "
      f"\"this answer is good\", it agrees with the catalog on "
      f"{q['accuracy_at_threshold']} of rows:")
print(f"  judge good, catalog hit  {c['tp']:3}     judge good, catalog miss {c['fp']:3}")
print(f"  judge poor, catalog hit  {c['fn']:3}     judge poor, catalog miss {c['tn']:3}")

## 9 · The judge's ceiling

The probe above measures the judge on our two systems. This one measures the
judge itself. Every reference answer is graded, then a corrupted copy of it is
graded too — an invented exercise name and an adaptation that would hurt someone
— and the pair is scored twice: once with the reference answer sitting in the
prompt, once without it.

That difference is the diagnosis. Separation with the reference and a collapse to
chance without it means the judge is comparing the answer against the reference
rather than evaluating it, and every score it gives a system whose reference it
cannot see should be read at the lower number. The AUC with the reference is also
an upper bound: no comparison between two real systems can be sharper than the
judge's ability to tell a gold answer from a deliberately broken copy of itself.

In [ ]:
print("with the reference answer in the prompt:")
ceiling_ref = H.judge_ceiling_probe(judge, eval_set, with_reference=True)
print("\nwithout it:")
ceiling_blind = H.judge_ceiling_probe(judge, eval_set, with_reference=False)

print(f"\nAUC, gold vs corrupted, reference shown : {ceiling_ref['auc']}"
      f"   (gold {ceiling_ref['mean_on_hits']} vs corrupted {ceiling_ref['mean_on_misses']})")
print(f"AUC, gold vs corrupted, reference hidden: {ceiling_blind['auc']}"
      f"   (gold {ceiling_blind['mean_on_hits']} vs corrupted {ceiling_blind['mean_on_misses']})")
print("\n0.500 is chance. Whatever the first number is, it is the ceiling on every "
      "judge comparison in this notebook.")

## 10 · The honest reading

The baseline is weak, and the more useful finding is that two of our three
dimensions failed to notice. In the run recorded in
`reports/scorecard_baseline.csv`, dimension 2 gave the zero-shot baseline a flat
5.000 across every column, including the adversarial block, and ranked it above
the fine-tuned model at 4.643 — while dimension 3, reading the same replies,
scored that same baseline 0 of 14. The two dimensions disagree in direction, not
in magnitude, and dimension 3 is the one we believe, because it is the only one
that resolves the answer against the catalog instead of reading it. The pooled
AUC in section 8 is where that disagreement stops being an anecdote.

The clearest single case is `adv04`. Asked for a lats exercise with a curtain
rail, the baseline told the user to hang from the curtain rail with their hands
slightly wider than their shoulders, and the judge scored it 5 — against a rubric
whose own standing rule is that anything which could injure the user caps the
score at 1. That is not a scoring artefact to be tuned away. It is a 1.5B model
failing to apply the hardest rule it was given, on the one case in the set where
being wrong hurts somebody, and it is the argument for carrying three dimensions
instead of one. The whole adversarial block is 0 of 4 for both systems: neither
refuses the impossible pair, corrects the false premise, declines the dosing
question, nor balks at the curtain rail. Fine-tuning moved the answerable cases
and left the refusals exactly where they were.

What M3 has to fix is grounding, not fluency. The model invents exercise names
because nothing in the loop obliges it to pick one that exists, and both metrics
that read the answer instead of resolving it were content to accept the
inventions. Retrieval over the reachable catalog entries turns "recall a name"
into "choose from this list", which should move dimension 3 first and carry the
ROUGE column with it. Refusal is the second target and the harder one: if
retrieval returns nothing for a pair the catalog cannot serve, the honest answer
becomes available without the model having to know, unaided, that the exercise is
absent. Both claims get defended against this exact scorecard — same eval set,
same rubric version, same seed.

In [ ]:
# One closing view: three dimensions, two systems, and the two bias probes.
cards = [sc_zeroshot, sc_lora]
ROWS = [
    ("1 · embedding similarity",         "similarity_mean"),
    ("1 · step grounding ROUGE-L",       "step_grounding_rouge_l"),
    ("2 · judge mean (1-5)",             "judge_mean"),
    ("2 · judge mean, adversarial",      "judge_mean_adversarial"),
    ("2 · judge vs dim 3, AUC",          "judge_vs_oracle_auc"),
    ("3 · domain hit rate",              "domain_hit_rate"),
    ("3 · domain hit rate, answerable",  "domain_hit_rate_gold"),
    ("3 · domain hit rate, adversarial", "domain_hit_rate_adversarial"),
]
cell = lambda v: "n/a" if v is None else f"{v:.3f}"

header = f"{'':36}" + "".join(f"{c['system']:>14}" for c in cards)
print(header)
print("-" * len(header))
for label, key in ROWS:
    print(f"{label:36}" + "".join(f"{cell(c.get(key)):>14}" for c in cards))

print("-" * len(header))
print(f"{'position bias, flip rate':36}{cell(position['flip_rate']):>14}")
print(f"{'length bias, mean delta uncapped':36}{cell(length['mean_delta_uncapped']):>14}")
print(f"{'length bias, mean delta capped':36}{cell(length['mean_delta_capped']):>14}")
print(f"{'judge vs dim 3, pooled AUC':36}{cell(q['auc']):>14}")
print("-" * len(header))
print(f"rubric {sc_lora['rubric_version']} · judge {sc_lora['judge_model']} · "
      f"seed {sc_lora['seed']} · {sc_lora['n']} examples")
print("per-system AUC is n/a where a system has no domain hits at all; the pooled "
      "row is the one to quote.")

## Reusing this in M3

```python
def rag_system(prompt_text):
    docs = retriever.search(prompt_text)
    return generator(prompt_text, context=docs)

sc_rag = H.harness(eval_set, rag_system, judge=judge, similarity=similarity, label="rag_m3")
H.write_scorecard([sc_zeroshot, sc_lora, sc_rag], "reports/scorecard_m3.csv")
```

Same eval set, same rubric version, same seed. If the rubric changes, bump the
version in `eval/rubric.md` and re-run every column — scores from two rubric
versions do not belong in one table.
